In [ ]:
#NOME: Tárcila Jéssica Silva de Sousa

In [ ]:
# TRABALHO 2

In [4]:
#TÓPICO 2
#1.1
using LinearAlgebra
using Printf

# Função:
# f(x1,x2,x3) = x3*log(exp(x1/x3) + exp(x2/x3)) + (x3-2)^2 + exp(1/(x1+x2))
#
# Domínio:
# x1 + x2 > 0
# x3 > 0
#
# Métodos:
# 1) Gradient Descent + Backtracking
# 2) Newton + Backtracking

# 1. Verificação do domínio

function in_domain(x::AbstractVector)
    return (x[1] + x[2] > 0.0) && (x[3] > 0.0)
end


# 2. Função objetivo

function f(x::AbstractVector)
    x1, x2, x3 = x
    return x3 * log(exp(x1 / x3) + exp(x2 / x3)) +
           (x3 - 2.0)^2 +
           exp(1.0 / (x1 + x2))
end


# 3. Gradiente analítico

function grad_f(x::AbstractVector)
    x1, x2, x3 = x

    a = exp(x1 / x3)
    b = exp(x2 / x3)
    S = a + b
    q = exp(1.0 / (x1 + x2))

    g1 = a / S - q / (x1 + x2)^2
    g2 = b / S - q / (x1 + x2)^2
    g3 = log(S) - (x1 * a + x2 * b) / (x3 * S) + 2.0 * (x3 - 2.0)

    return [g1, g2, g3]
end


# 4. Hessiana por diferenças finitas
#    H[:,j] ≈ (grad(x + h*e_j) - grad(x)) / h

function hessian_fd(grad_f, x; h=1e-6)
    n = length(x)
    H = zeros(n, n)
    g0 = grad_f(x)

    for j in 1:n
        e = zeros(n)
        e[j] = 1.0
        H[:, j] = (grad_f(x .+ h .* e) - g0) / h
    end

    return 0.5 * (H + H')   # força simetria numérica
end


# 5. Backtracking line search
#    Parâmetros do enunciado:
#    tinit = 1, α = 0.4, β = 0.5

function backtracking(f, grad_f, x, d; α=0.4, β=0.5, tinit=1.0, tmin=1e-20)
    t = tinit
    fx = f(x)
    gx = grad_f(x)

    while true
        x_new = x .+ t .* d

        # Aceita apenas se permanecer no domínio
        # e satisfizer a condição de Armijo
        if in_domain(x_new) && f(x_new) <= fx + α * t * dot(gx, d)
            return t
        end

        t *= β

        if t < tmin
            error("Backtracking falhou: passo ficou pequeno demais.")
        end
    end
end


# 6. Gradient Descent

function gradient_descent(f, grad_f, x0; tol=1e-5, maxit=1000)
    if !in_domain(x0)
        error("O ponto inicial não pertence ao domínio.")
    end

    x = copy(x0)

    println("GRADIENT DESCENT")
    @printf("%3s %12s %12s %12s %14s %12s\n",
            "k", "x1", "x2", "x3", "f(x)", "||grad||")

    for k in 0:maxit
        g = grad_f(x)
        ng = norm(g)
        fx = f(x)

        @printf("%3d %12.6f %12.6f %12.6f %14.8f %12.4e\n",
                k, x[1], x[2], x[3], fx, ng)

        if ng < tol
            println("\nConvergiu pelo critério ||grad|| < $tol")
            return x
        end

        d = -g
        t = backtracking(f, grad_f, x, d)
        x = x .+ t .* d
    end

    println("\nNúmero máximo de iterações atingido.")
    return x
end


# 7. Método de Newton

function newton_method(f, grad_f, x0; tol=1e-5, maxit=100)
    if !in_domain(x0)
        error("O ponto inicial não pertence ao domínio.")
    end

    x = copy(x0)

    println("MÉTODO DE NEWTON")
    @printf("%3s %12s %12s %12s %14s %12s\n",
            "k", "x1", "x2", "x3", "f(x)", "||grad||")

    for k in 0:maxit
        g = grad_f(x)
        ng = norm(g)
        fx = f(x)

        @printf("%3d %12.6f %12.6f %12.6f %14.8f %12.4e\n",
                k, x[1], x[2], x[3], fx, ng)

        if ng < tol
            println("\nConvergiu pelo critério ||grad|| < $tol")
            return x
        end

        H = hessian_fd(grad_f, x)

        # direção de Newton
        d = -(H \ g)

        # proteção: se não for direção de descida, usa -gradiente
        if dot(g, d) >= 0
            d = -g
        end

        t = backtracking(f, grad_f, x, d)
        x = x .+ t .* d
    end

    println("\nNúmero máximo de iterações atingido.")
    return x
end

x0 = [1.0, 1.0, 1.0]   # ponto inicial factível

println("Ponto inicial: ", x0)
println("f(x0) = ", f(x0))
println("grad_f(x0) = ", grad_f(x0))
println()

# Gradient Descent
x_gd = gradient_descent(f, grad_f, x0; tol=1e-5, maxit=1000)

println("\nResultado final - Gradient Descent")
println("x* = ", x_gd)
println("f(x*) = ", f(x_gd))
println("||grad_f(x*)|| = ", norm(grad_f(x_gd)))

# Newton
x_nt = newton_method(f, grad_f, x0; tol=1e-5, maxit=100)

println("\nResultado final - Newton")
println("x* = ", x_nt)
println("f(x*) = ", f(x_nt))
println("||grad_f(x*)|| = ", norm(grad_f(x_nt)))

Ponto inicial: [1.0, 1.0, 1.0]
f(x0) = 4.341868451260074
grad_f(x0) = [0.08781968232496795, 0.08781968232496795, -1.3068528194400548]

GRADIENT DESCENT
  k           x1           x2           x3           f(x)     ||grad||
  0     1.000000     1.000000     1.000000     4.34186845   1.3127e+00
  1     0.956090     0.956090     1.653426     3.90929055   5.4612e-02
  2     0.936782     0.936782     1.653426     3.90826492   2.0071e-02
  3     0.929686     0.929686     1.653426     3.90813029   6.6953e-03
  4     0.927319     0.927319     1.653426     3.90811547   2.1477e-03
  5     0.926559     0.926559     1.653426     3.90811395   6.7971e-04
  6     0.926319     0.926319     1.653426     3.90811380   2.1417e-04
  7     0.926243     0.926243     1.653426     3.90811379   6.7391e-05
  8     0.926219     0.926219     1.653426     3.90811379   2.1196e-05
  9     0.926212     0.926212     1.653426     3.90811379   6.6655e-06

Convergiu pelo critério ||grad|| < 1.0e-5

Resultado final - Gradi

In [5]:
#TÓPICO 2
#1.2

using LinearAlgebra
using Printf

function in_domain(x::AbstractVector)
    return (x[1] + x[2] > 0.0) && (x[3] > 0.0)
end
function f(x::AbstractVector)
    x1, x2, x3 = x
    return x3 * log(exp(x1 / x3) + exp(x2 / x3)) +
           (x3 - 2.0)^2 +
           exp(1.0 / (x1 + x2))
end

function grad_f(x::AbstractVector)
    x1, x2, x3 = x

    a = exp(x1 / x3)
    b = exp(x2 / x3)
    S = a + b
    q = exp(1.0 / (x1 + x2))

    g1 = a / S - q / (x1 + x2)^2
    g2 = b / S - q / (x1 + x2)^2
    g3 = log(S) - (x1 * a + x2 * b) / (x3 * S) + 2.0 * (x3 - 2.0)

    return [g1, g2, g3]
end

function backtracking(f, grad_f, x, d; α=0.4, β=0.5, tinit=1.0, tmin=1e-20)
    t = tinit
    fx = f(x)
    gx = grad_f(x)

    while true
        x_new = x .+ t .* d

        if in_domain(x_new) && f(x_new) <= fx + α * t * dot(gx, d)
            return t
        end

        t *= β

        if t < tmin
            error("Backtracking falhou: passo ficou pequeno demais.")
        end
    end
end


# Método BFGS
#    H aproxima a inversa da Hessiana
#    H0 = I

function bfgs_method(f, grad_f, x0; tol=1e-5, maxit=500)
    if !in_domain(x0)
        error("O ponto inicial não pertence ao domínio.")
    end

    x = copy(x0)
    n = length(x)
    H = Matrix{Float64}(I, n, n)   # H0 = I

    println("MÉTODO BFGS")
    @printf("%3s %12s %12s %12s %14s %12s\n",
            "k", "x1", "x2", "x3", "f(x)", "||grad||")

    for k in 0:maxit
        g = grad_f(x)
        ng = norm(g)
        fx = f(x)

        @printf("%3d %12.6f %12.6f %12.6f %14.8f %12.4e\n",
                k, x[1], x[2], x[3], fx, ng)

        if ng < tol
            println("\nConvergiu pelo critério ||grad|| < $tol")
            return x
        end

        # direção de busca
        d = -H * g

        # proteção: se não for direção de descida, usa -gradiente
        if dot(g, d) >= 0
            d = -g
        end

        # line search
        t = backtracking(f, grad_f, x, d; α=0.4, β=0.5, tinit=1.0)

        x_new = x .+ t .* d
        g_new = grad_f(x_new)

        s = x_new - x
        y = g_new - g
        ys = dot(y, s)

        # atualização BFGS segura
        if ys > 1e-5
            ρ = 1.0 / ys
            I_n = Matrix{Float64}(I, n, n)
            V = I_n - ρ * (s * y')
            H = V * H * V' + ρ * (s * s')
        end

        x = x_new
    end

    println("\nNúmero máximo de iterações atingido.")
    return x
end

x0 = [1.0, 1.0, 1.0]

println("Ponto inicial: ", x0)
println("f(x0) = ", f(x0))
println("grad_f(x0) = ", grad_f(x0))
println()

x_bfgs = bfgs_method(f, grad_f, x0; tol=1e-5, maxit=500)

println("\nResultado final - BFGS")
println("x* = ", x_bfgs)
println("f(x*) = ", f(x_bfgs))
println("||grad_f(x*)|| = ", norm(grad_f(x_bfgs)))

Ponto inicial: [1.0, 1.0, 1.0]
f(x0) = 4.341868451260074
grad_f(x0) = [0.08781968232496795, 0.08781968232496795, -1.3068528194400548]

MÉTODO BFGS
  k           x1           x2           x3           f(x)     ||grad||
  0     1.000000     1.000000     1.000000     4.34186845   1.3127e+00
  1     0.956090     0.956090     1.653426     3.90929055   5.4612e-02
  2     0.936717     0.936717     1.653265     3.90826309   1.9952e-02
  3     0.925566     0.925566     1.653427     3.90811435   1.2472e-03
  4     0.926222     0.926222     1.653426     3.90811379   2.6577e-05
  5     0.926208     0.926208     1.653426     3.90811379   5.3104e-07

Convergiu pelo critério ||grad|| < 1.0e-5

Resultado final - BFGS
x* = [0.9262081852498661, 0.9262081852498661, 1.6534264096134241]
f(x*) = 3.90811378626197
||grad_f(x*)|| = 5.310438031431418e-7


In [6]:
#TÓPICO 2
#1.3

using JuMP
using HiGHS

# =========================================================
# ITEM 1.3 - FORMULAÇÃO LP DO REATOR CSTR
# =========================================================

# Matrizes do modelo
A = [0.2681   -0.00338   -0.00728;
     9.703     0.3279    -25.44;
     0.0       0.0        1.0]

B = [-0.00537   0.1655;
      1.297    97.91;
      0.0      -6.637]

# Horizonte: k = 0,1,2,3
N = 3

# Modelo
model = Model(HiGHS.Optimizer)

# Variáveis de estado: x[1:3, 0:N]
@variable(model, x[1:3, 0:N])

# Variáveis de controle: u[1:2, 0:N-1]
@variable(model, u[1:2, 0:N-1])

# Variáveis auxiliares para |x1| e |x3|
@variable(model, p[0:N] >= 0)   # aproxima |x1(k)|
@variable(model, q[0:N] >= 0)   # aproxima |x3(k)|

# ---------------------------------------------------------
# Condição inicial
# ---------------------------------------------------------
@constraint(model, x[1,0] == -0.03)
@constraint(model, x[2,0] ==  0.0)
@constraint(model, x[3,0] ==  0.3)

# ---------------------------------------------------------
# Dinâmica do sistema
# x(k+1) = A*x(k) + B*u(k)
# ---------------------------------------------------------
for k in 0:N-1
    @constraint(model,
        x[1,k+1] == 0.2681*x[1,k] - 0.00338*x[2,k] - 0.00728*x[3,k]
                    - 0.00537*u[1,k] + 0.1655*u[2,k])

    @constraint(model,
        x[2,k+1] == 9.703*x[1,k] + 0.3279*x[2,k] - 25.44*x[3,k]
                    + 1.297*u[1,k] + 97.91*u[2,k])

    @constraint(model,
        x[3,k+1] == x[3,k] - 6.637*u[2,k])
end

# ---------------------------------------------------------
# Restrições de estado
# ---------------------------------------------------------
for k in 0:N
    @constraint(model, -0.05 <= x[1,k] <= 0.05)
    @constraint(model, -5.0  <= x[2,k] <= 5.0)
    @constraint(model, -0.5  <= x[3,k] <= 0.5)
end

# ---------------------------------------------------------
# Restrições de entrada
# ---------------------------------------------------------
for k in 0:N-1
    @constraint(model, -10.0 <= u[1,k] <= 10.0)
    @constraint(model, -0.05 <= u[2,k] <= 0.05)
end

# ---------------------------------------------------------
# Linearização dos módulos:
# p[k] >= |x1(k)|, q[k] >= |x3(k)|
# ---------------------------------------------------------
for k in 0:N
    @constraint(model, p[k] >=  x[1,k])
    @constraint(model, p[k] >= -x[1,k])

    @constraint(model, q[k] >=  x[3,k])
    @constraint(model, q[k] >= -x[3,k])
end

# ---------------------------------------------------------
# Função objetivo
# min Σ ( |x1(k)| + |x3(k)| ) ≈ Σ (p[k] + q[k])
# ---------------------------------------------------------
@objective(model, Min, sum(p[k] + q[k] for k in 0:N))

# Resolver
optimize!(model)

# Resultados
println("Status = ", termination_status(model))
println("Valor ótimo = ", objective_value(model))
println()

println("Estados ótimos:")
for k in 0:N
    println("x($k) = ", [value(x[i,k]) for i in 1:3])
end

println()
println("Controles ótimos:")
for k in 0:N-1
    println("u($k) = ", [value(u[i,k]) for i in 1:2])
end





Running HiGHS 1.14.0 (git hash: 7df0786de3): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline-5 
LP has 46 rows; 26 cols; 98 nonzeros
Coefficient ranges:
  Matrix  [3e-03, 1e+02]
  Cost    [1e+00, 1e+00]
  Bound   [0e+00, 0e+00]
  RHS     [3e-02, 1e+01]
Presolving model
20 rows, 20 cols, 60 nonzeros 0s
Dependent equations search running on 7 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
19 rows, 19 cols, 57 nonzeros 0s
Presolve reductions: rows 19(-27); columns 19(-7); nonzeros 57(-41) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     3.2999937732e-01 Pr: 13(6.69911) 0.0s
         16     3.3000000000e-01 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Simplex   iterations: 16
Objective value     :  3.3000000000e-01
P-D objective erro